# ConnectomePrune-LLM: Reproducible Benchmark Engine
### Attention-Protective Non-Uniform Sparsification for Compact Language Models
**Double-Blind Review Reproducibility Notebook**

This notebook provides an automated reproduction suite for **ConnectomePrune-LLM** on **LLaMA-3.2-1B** and **Qwen2.5-1.5B**.

> **Hardware & Precision Note:** Standard Google Colab T4 GPUs (16 GB VRAM) lack native bfloat16 hardware support; execution should use float16 (`torch.float16`) for stable high throughput on 1B and 1.5B architectures. The offline tensor carving tool operates entirely on CPU with < 3.5 GB peak host RAM.

In [ ]:
# Step 1: Install Dependencies
!pip install -q torch transformers accelerate datasets safetensors lm_eval human-eval matplotlib scipy

In [ ]:
# Step 2: Hugging Face Authentication (Required for Meta LLaMA gated models)
from huggingface_hub import login
# Paste your HF read token below, or skip if testing open-weight models (e.g. Qwen/Qwen2.5-1.5B)
# login('YOUR_HF_TOKEN_HERE')

In [ ]:
# Step 3: Setup Codebase Environment
# For double-blind review: unzip the supplementary archive or install directly
# (e.g. !unzip -q supplementary_code.zip && cd connectome-prune-llm)
!pip install -q -e .

In [ ]:
# Step 4: Run Main Benchmark Suite (LLaMA-3.2-1B, Regime A WikiText-2)
# Evaluates Dense Baseline, Wanda, SparseGPT, OWL, ConnectomePrune, and Inverse-Hub controls
!python run_q1_benchmarks.py

In [ ]:
# Step 5: Run OWL Integration & Regime B Zero-Shot Stress Protocol
!python run_owl_hub_scaling_ablation.py

In [ ]:
# Step 6: Run Module-Specific Homeostatic Synaptic Scaling Sweep
# Evaluates Dense, Hub-Only, Naive Uniform, W_o-only, W_down-only, and Combined module scaling
!python run_gamma_sweep.py

In [ ]:
# Step 7: Statistical Analysis & Hypothesis Testing
import math
from scipy.stats import binomtest

def wilson_ci(k, n, z=1.95996):
    p = k / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    rad = (z / denom) * math.sqrt(p * (1 - p) / n + z**2 / (4 * n**2))
    return round(max(0.0, center - rad) * 100, 1), round(min(1.0, center + rad) * 100, 1)

print('Exact 95% Wilson CIs for Code Syntax (n=50):')
for k, name in [(50, 'ConnectomePrune / Dense'), (49, 'OWL + Scaling'), (48, 'OWL / Wanda+Hub'), (40, 'Wanda')]:
    lo, hi = wilson_ci(k, 50)
    print(f'  {name:<25}: {k}/50 ({k/50*100:.1f}%) -> [{lo}, {hi}]')

print('\nExact Binomial McNemar Tests:')
# ConnectomePrune (50/50) vs Wanda (40/50): 10 discordant pairs
res_wanda = binomtest(10, 10, 0.5, alternative='two-sided')
print(f'  ConnectomePrune vs. Wanda: p = {res_wanda.pvalue:.5f} (Statistically Significant Separation, p < 0.005)')
# ConnectomePrune (50/50) vs OWL+Scaling (49/50): 1 discordant pair
res_owl = binomtest(1, 1, 0.5, alternative='two-sided')
print(f'  ConnectomePrune vs. OWL+Scaling: p = {res_owl.pvalue:.4f} (Statistically Indistinguishable, Parity confirmed)')


In [ ]:
# Step 8: Generate Mechanistic Layer-Wise Saliency Analysis
!python generate_mechanistic_analysis.py
!python plot_layer_saliency.py
from IPython.display import Image, display
display(Image('figures/fig1_layer_saliency_ratio.png'))